# Greedy text generation utilities

This utility notebook is the nbdev source of truth for generation helpers used by Chapter 4 and later chapters.

In [ ]:
#| default_exp generation

In [ ]:
#| export
import torch

from build_llms_from_scratch_companion.model import GPTModel

## Generating token IDs one position at a time

`generate_text_simple` crops the input to the model's `context_length`, selects the logits at the final input position, chooses the
highest-probability token ID, and appends it to the sequence. Shapes are documented at each operation because the token dimension
grows by one on every iteration.

In [ ]:
#| export
def generate_text_simple(
    model: GPTModel,
    idx: torch.Tensor,
    max_new_tokens: int,
    context_size: int,
) -> torch.Tensor:
    """Append greedily selected token IDs to a batch of input token IDs.

    Args:
        model: GPT model used to produce next-token logits.
        idx: Input token IDs shaped `(batch_size, num_tokens)`.
        max_new_tokens: Number of token IDs to generate and append.
        context_size: Maximum number of recent token IDs supplied to the model.

    Returns:
        Token IDs shaped
        `(batch_size, num_tokens + max_new_tokens)`.
    """
    for _ in range(max_new_tokens):
        # idx: (batch_size, num_tokens)
        # idx_cond: (batch_size, min(num_tokens, context_size))
        idx_cond = idx[:, -context_size:]
        # Generation is inference-only, so no backward graph is required.
        with torch.no_grad():
            # logits: (batch_size, min(num_tokens, context_size), vocab_size)
            logits = model(idx_cond)

        # Keep the next-token logits from the final input position only.
        # logits: (batch_size, vocab_size)
        logits = logits[:, -1, :]
        # probas: (batch_size, vocab_size)
        probas = torch.softmax(logits, dim=-1)
        # idx_next: (batch_size, 1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        # idx: (batch_size, num_tokens + 1)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx